# ETL: Pipelines for Complex Data 🔄


- Process complex PDFs containing text and tables using the `unstructured` library.
- Apply intelligent chunking strategies with `RecursiveCharacterTextSplitter`.
- Ingest data from an SQL database with `DuckDB` and transform it into `Documents`.
- Enrich our `Documents` with strategic metadata to enhance searches.

### Why are data pipelines essential for RAG?
- **Context Quality**: The way you process and split your data (chunking) directly affects the context the LLM receives, and therefore, the quality of the response.
- **Source Diversity**: Production RAG systems feed on multiple sources: PDFs, databases, APIs, etc.
- **Metadata**: They are the key to filtered searches, traceability, and access control, making your RAG much more powerful.

## Settings

Let's install the necessary libraries. Note that `unstructured` may have additional dependencies to process certain file types.

**Attention:** Installing `unstructured` may take some time.

In [ ]:
!pip install langchain langchain-google-genai "unstructured[pdf]" duckdb pandas
!pip install langchain_community
!pip install chromadb
!pip install pdfminer.six
!pip install --upgrade --force-reinstall pdfminer.six unstructured

### Importing Libraries

In [ ]:
import os
import duckdb
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
import os

### Setup - Load API Key and Initialize Client

In [ ]:
# Load the .env file
load_dotenv(dotenv_path='../../.env')  # Specify the path to your .env file

# Access the environment variable
api_key = os.getenv('OPENAI_API_KEY')

# Check if the variable is loaded
if api_key or api_key == "":
    print("API key loaded successfully.")
else:
    print("Failed to load API key.")

from openai import OpenAI
client = OpenAI(api_key=api_key)

## 1. Processing Complex PDFs with `Unstructured`

Unlike `PyPDFLoader`, the `UnstructuredPDFLoader` is designed to understand the structure of a PDF, such as titles, paragraphs, and, crucially, **tables**. It attempts to extract each element separately, which is excellent for chunking.

We will use the `relatorio_vendas.pdf` we created, which contains text and a table.

In [ ]:
from langchain_community.document_loaders import UnstructuredPDFLoader

pdf_path = "../../data/sales_report.pdf"
loader = UnstructuredPDFLoader(pdf_path, mode="elements")

docs_unstructured = loader.load()

print(f"Total elements extracted: {len(docs_unstructured)}\n")

for doc in docs_unstructured:
  print(f"---- ELEMENT TYPE: {doc.metadata.get('category')} ---")
  print(doc.page_content)
  print("\n")

### Adding Strategic Metadata During Loading

It is good practice to add metadata at the time of data loading.

In [ ]:
from langchain.schema.document import Document

docs_with_metadata = []

for doc in docs_unstructured:

  new_metadata = doc.metadata.copy()

  new_metadata['source'] = pdf_path
  new_metadata['ingestion_date'] = datetime.now().strftime('%Y-%m-%d')
  new_metadata['data_owner'] = 'Sales Team'

  docs_with_metadata.append(
      Document(page_content=doc.page_content, metadata=new_metadata)
  )

print(f"Total: {len(docs_with_metadata)}\n")
print(docs_with_metadata[-1])

## 2. Smart Chunking with  `RecursiveCharacterTextSplitter`


In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap  = 50,
)

chunks = text_splitter.split_documents(docs_with_metadata)

print(f"Original: {len(docs_with_metadata)}")
print(f"Generated: {len(chunks)}\n")

print(chunks[2])

## 3. Loading data from a Dabase - `DuckDB`

In [ ]:
import duckdb
import pandas as pd

# Connecting to an in-memory DuckDB instance
con = duckdb.connect(database=':memory:', read_only=False)

# crate a sample table
con.execute("""
CREATE TABLE products (
    id INTEGER,
    name VARCHAR,
    category VARCHAR,
    price FLOAT,
    stock INTEGER,
    description VARCHAR
);
""")

# Insert sample data
products_df = pd.DataFrame({
    'id': [101, 102, 103, 104],
    'name': ['Laptop Gamer Z', 'Fast Optical Mouse', 'Pro Mechanical Keyboard', '34" Curved Monitor'],
    'category': ['Electronics', 'Accessories', 'Accessories', 'Electronics'],
    'price': [9500.00, 250.00, 800.00, 3200.00],
    'stock': [15, 120, 60, 25],
    'description': [
        'High-performance laptop with dedicated graphics card and 32GB RAM.',
        'Mouse with 16,000 DPI and ergonomic design for long sessions.',
        'Keyboard with mechanical switches, RGB lighting, and ABNT2 layout.',
        'Ultrawide monitor with high refresh rate and vibrant colors.'
    ]
})

con.register('products_df', products_df)
con.execute('INSERT INTO products SELECT * FROM products_df;')

print("Table products created.")

# check the data
print(con.execute("SELECT * FROM products;").fetchdf())


### Converting SQL Data into `Documents`

In [ ]:

from langchain.schema.document import Document

# select data from the table
df_products = con.execute("SELECT * FROM products;").fetchdf()

# list to hold documents
docs_sql = []

for _, row in products_df.iterrows():
    # create a textual representation of the row
    page_content = f"Product: {row['name']}. Category: {row['category']}. Price: ${row['price']:.2f}. In stock: {row['stock']} units. Description: {row['description']}"

    # metadata for the document
    metadata = {
        'source': 'products_table_duckdb',
        'product_id': row['id'],
        'category': row['category'],
        'price': row['price'],
        'ingestion_date': datetime.now().strftime('%Y-%m-%d')
    }

    docs_sql.append(
        Document(page_content=page_content, metadata=metadata)
    )

# close the connection
con.close()

print(f"Total documents generated from SQL: {len(docs_sql)}\n")
print("Example of a document generated from a database row:")
print(docs_sql[0])


## 4. Merging Pipelines and Sending to the Vector Store

Now we have `chunks` from the PDF and `docs_sql` from the database. The next step is to merge them and send them to a vector database (such as Chroma, FAISS, or Pinecone) for indexing.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.vectorstores.utils import filter_complex_metadata

final_docs = chunks + docs_sql

print(f"Total docs to indexing: {len(final_docs)}")

filtered_docs = filter_complex_metadata(final_docs)

print(f"Total filtered docs: {len(filtered_docs)}")

embeddings = OpenAIEmbeddings(openai_api_key=api_key)

vector_store = Chroma.from_documents(
    documents=filtered_docs,
    embedding=embeddings
)


### Testing the Final Result

Let's perform a search to see if our RAG can find information from both the PDF and the database.

In [ ]:
doc.metadata

In [ ]:
# Question about the PDF
document_query_pdf = "What was the revenue from laptops?"

pdf_results = vector_store.similarity_search(document_query_pdf, k=2)

print(f"Question: {document_query_pdf}\n")

for doc in pdf_results:
  print(f"- Similarity: {doc.page_content} ")
  print(f" (Source: {doc.metadata.get('source')}, Category: {doc.metadata.get('category')})")

print("-"*20)

# Question about the Database
document_query_sql = "Tell me about the mechanical keyboard"
sql_results = vector_store.similarity_search(document_query_sql, k=2)

print(f"Question: {document_query_sql}\n")
for doc in sql_results:
  print(f"- Similarity: {doc.page_content} ")
  print(f"    (Source: {doc.metadata.get('source')}, Category: {doc.metadata.get('category')})")


## 📚 Resumo Prático da Aula 4

- **Use a ferramenta certa**: `UnstructuredPDFLoader` é superior ao `PyPDFLoader` para documentos com estruturas complexas como tabelas.
- **Metadados são seu melhor amigo**: Enriquecer os documentos durante a ingestão com informações de fonte, datas e outros atributos é o que permite criar RAGs realmente úteis e confiáveis.
- **RAG não é só para texto**: Transformar dados estruturados (de SQL, CSVs, etc.) em documentos textuais expande enormemente o conhecimento do seu sistema.
- **Pipeline é um processo**: O fluxo `Load -> Transform (Add Metadata) -> Split -> Index` é um padrão robusto para a maioria das necessidades de ingestão de dados em RAG.
